# Usage: load artifact & predict

Keep the kernel **working directory** as `models/` (same as `main.ipynb`) so `./artifacts/big5_ridge.joblib` resolves. If the file is missing, run the **Persist the chosen model** cell in `main.ipynb` first.


In [1]:
"""Load Ridge bundle and rebuild inference helpers (same logic as main.ipynb)."""

from pathlib import Path

import joblib
import numpy as np

MODEL_DIR = Path(".").resolve()
ARTIFACT_PATH = MODEL_DIR / "artifacts" / "big5_ridge.joblib"

if not ARTIFACT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {ARTIFACT_PATH}. Run the persist cell in main.ipynb first."
    )

_bundle = joblib.load(ARTIFACT_PATH)
ridge = _bundle["ridge"]
ALPHA = _bundle["alpha"]
ALL_Q_COLS = list(_bundle["all_q_cols"])
TRAIT_COLS = dict(_bundle["trait_cols"])
MEME_QUESTIONS = list(_bundle["meme_questions"])

COL_TO_IDX = {c: i for i, c in enumerate(ALL_Q_COLS)}
TRAIT_IDXS = {
    trait: np.array([COL_TO_IDX[c] for c in qs], dtype=np.int64)
    for trait, qs in TRAIT_COLS.items()
}

print(f"Loaded Ridge (alpha={ALPHA}), {len(ALL_Q_COLS)} questions, {len(MEME_QUESTIONS)} meme items")


def make_features(X_masked: np.ndarray, mask: np.ndarray) -> np.ndarray:
    return np.hstack([X_masked, mask]).astype(np.float32)


def predict_ridge(X_masked: np.ndarray, mask: np.ndarray) -> np.ndarray:
    pred = ridge.predict(make_features(X_masked, mask))
    pred = np.where(mask == 1, X_masked, pred)
    return np.clip(pred, 1, 5)


def trait_scores_from_answers(X: np.ndarray) -> np.ndarray:
    """(n, 50) item scores -> (n, 5) trait means on the 1..5 survey scale."""
    out = np.empty((len(X), len(TRAIT_IDXS)), dtype=np.float32)
    for ti, idxs in enumerate(TRAIT_IDXS.values()):
        out[:, ti] = X[:, idxs].mean(axis=1)
    return out


def predict_personality(answers: dict[str, float]) -> dict:
    """Partial answers {question_code: 1..5} -> full quiz result payload (cf. main.ipynb)."""
    x = np.zeros((1, len(ALL_Q_COLS)), dtype=np.float32)
    m = np.zeros((1, len(ALL_Q_COLS)), dtype=np.float32)
    for q, v in answers.items():
        if q in COL_TO_IDX:
            x[0, COL_TO_IDX[q]] = float(v)
            m[0, COL_TO_IDX[q]] = 1.0

    pred = predict_ridge(x, m)[0]
    traits = trait_scores_from_answers(pred[None, :])[0]
    answered_set = set(answers)

    def split(q_codes):
        ans, prd = {}, {}
        for q in q_codes:
            val = round(float(pred[COL_TO_IDX[q]]), 2)
            (ans if q in answered_set else prd)[q] = val
        return {"answered": ans, "predicted": prd}

    secondary_qs = [q for q in ALL_Q_COLS if q not in MEME_QUESTIONS]

    return {
        "n_answered": int(m.sum()),
        "trait_scores": dict(zip(TRAIT_COLS.keys(), traits.round(3).tolist())),
        "primary": split(MEME_QUESTIONS),
        "secondary": split(secondary_qs),
    }


# Fixed demo — same smoke test as main.ipynb (5 answers incl. CSN6)
demo = {"EXT3": 4, "EXT2": 2, "OPN3": 3, "CSN6": 1, "AGR2": 4}
result = predict_personality(demo)
assert result["n_answered"] == 5
assert len(result["primary"]["answered"]) + len(result["primary"]["predicted"]) == 7
assert len(result["secondary"]["answered"]) + len(result["secondary"]["predicted"]) == 43

print("Sanity OK — fixed demo:")
print(f"  n_answered={result['n_answered']}")
print(f"  trait_scores={result['trait_scores']}")
print(
    "  primary   known/pred ="
    f" {len(result['primary']['answered'])}/{len(result['primary']['predicted'])}"
)
print(
    "  secondary known/pred ="
    f" {len(result['secondary']['answered'])}/{len(result['secondary']['predicted'])}"
)
result

Loaded Ridge (alpha=1600.0), 50 questions, 7 meme items
Sanity OK — fixed demo:
  n_answered=5
  trait_scores={'EXT': 2.996000051498413, 'EST': 2.9839999675750732, 'AGR': 3.1480000019073486, 'CSN': 2.9189999103546143, 'OPN': 3.1449999809265137}
  primary   known/pred = 2/5
  secondary known/pred = 3/40


{'n_answered': 5,
 'trait_scores': {'EXT': 2.996000051498413,
  'EST': 2.9839999675750732,
  'AGR': 3.1480000019073486,
  'CSN': 2.9189999103546143,
  'OPN': 3.1449999809265137},
 'primary': {'answered': {'CSN6': 1.0, 'OPN3': 3.0},
  'predicted': {'AGR9': 3.77,
   'AGR5': 2.22,
   'EXT4': 3.01,
   'EST7': 3.01,
   'OPN10': 3.83}},
 'secondary': {'answered': {'EXT2': 2.0, 'EXT3': 4.0, 'AGR2': 4.0},
  'predicted': {'EXT1': 2.7,
   'EXT5': 3.38,
   'EXT6': 2.28,
   'EXT7': 2.85,
   'EXT8': 3.35,
   'EXT9': 2.99,
   'EXT10': 3.41,
   'EST1': 3.24,
   'EST2': 3.16,
   'EST3': 3.79,
   'EST4': 2.65,
   'EST5': 2.79,
   'EST6': 2.79,
   'EST8': 2.64,
   'EST9': 3.06,
   'EST10': 2.7,
   'AGR1': 2.22,
   'AGR3': 2.24,
   'AGR4': 3.91,
   'AGR6': 3.69,
   'AGR7': 2.1,
   'AGR8': 3.7,
   'AGR10': 3.61,
   'CSN1': 3.37,
   'CSN2': 2.66,
   'CSN3': 3.99,
   'CSN4': 2.43,
   'CSN5': 2.73,
   'CSN7': 3.77,
   'CSN8': 2.4,
   'CSN9': 3.26,
   'CSN10': 3.59,
   'OPN1': 3.61,
   'OPN2': 2.07,
   'OPN4'

### Playground

Edit `my_answers` (any subset of the 50 codes: `EXT1` … `OPN10`) and re-run the cell.


In [6]:
# Play — tweak values 1..5
my_answers = {
    "EXT2": 5,
}

out = predict_personality(my_answers)

print("Trait means (1–5):", out["trait_scores"])
print("\nMeme tier (headline):")
print("  answered:  ", out["primary"]["answered"])
print("  predicted:", out["primary"]["predicted"])
print("\nSample secondary (first 10 predicted keys):")
pred_sec = out["secondary"]["predicted"]
for i, (k, v) in enumerate(sorted(pred_sec.items())):
    if i >= 10:
        print(f"  ... ({len(pred_sec) - 10} more)")
        break
    print(f"  {k}: {v}")

out

Trait means (1–5): {'EXT': 3.2669999599456787, 'EST': 3.0190000534057617, 'AGR': 3.1570000648498535, 'CSN': 3.125999927520752, 'OPN': 3.2709999084472656}

Meme tier (headline):
  answered:   {}
  predicted: {'CSN6': 2.81, 'AGR9': 3.79, 'AGR5': 2.31, 'EXT4': 3.35, 'EST7': 3.03, 'OPN3': 4.0, 'OPN10': 3.95}

Sample secondary (first 10 predicted keys):
  AGR1: 2.3
  AGR10: 3.57
  AGR2: 3.81
  AGR3: 2.21
  AGR4: 3.93
  AGR6: 3.73
  AGR7: 2.24
  AGR8: 3.67
  CSN1: 3.32
  CSN10: 3.6
  ... (32 more)


{'n_answered': 1,
 'trait_scores': {'EXT': 3.2669999599456787,
  'EST': 3.0190000534057617,
  'AGR': 3.1570000648498535,
  'CSN': 3.125999927520752,
  'OPN': 3.2709999084472656},
 'primary': {'answered': {},
  'predicted': {'CSN6': 2.81,
   'AGR9': 3.79,
   'AGR5': 2.31,
   'EXT4': 3.35,
   'EST7': 3.03,
   'OPN3': 4.0,
   'OPN10': 3.95}},
 'secondary': {'answered': {'EXT2': 5.0},
  'predicted': {'EXT1': 2.54,
   'EXT3': 3.2,
   'EXT5': 3.1,
   'EXT6': 2.67,
   'EXT7': 2.68,
   'EXT8': 3.51,
   'EXT9': 2.91,
   'EXT10': 3.71,
   'EST1': 3.27,
   'EST2': 3.21,
   'EST3': 3.83,
   'EST4': 2.67,
   'EST5': 2.83,
   'EST6': 2.81,
   'EST8': 2.66,
   'EST9': 3.05,
   'EST10': 2.82,
   'AGR1': 2.3,
   'AGR2': 3.81,
   'AGR3': 2.21,
   'AGR4': 3.93,
   'AGR6': 3.73,
   'AGR7': 2.24,
   'AGR8': 3.67,
   'AGR10': 3.57,
   'CSN1': 3.32,
   'CSN2': 2.92,
   'CSN3': 4.0,
   'CSN4': 2.6,
   'CSN5': 2.65,
   'CSN7': 3.71,
   'CSN8': 2.48,
   'CSN9': 3.19,
   'CSN10': 3.6,
   'OPN1': 3.66,
   'OPN2':